In [1]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

In [2]:
region_decline_df = pd.read_csv('./ref_dataset/01_지역소멸지수.csv')
empty_index_df = pd.read_csv('./ref_dataset/02_폐가지수.csv')
visitors_df = pd.read_csv('./ref_dataset/04_방문자수.csv')

w1 = 5
w2 = 3
w3 = -1


In [3]:
temp_df1 = region_decline_df[['도시레벨','소멸위험도']]
temp_df2 = empty_index_df[['도시레벨', '페가지수_scaled']]
temp_df3 = visitors_df[['결합행정구역', 'visitors_scaled']]

temp_df3 = temp_df3.rename(columns={'결합행정구역': '도시레벨'})


In [4]:
merged_df = pd.merge(temp_df1, temp_df2, on='도시레벨', how='inner')
merged_df = pd.merge(merged_df, temp_df3, on='도시레벨', how='inner')

In [5]:
merged_df

,도시레벨,소멸위험도,페가지수_scaled,visitors_scaled
0,경기도 가평군,0.298097,0.275046,0.094243
1,경기도 과천시,0.029580,0.014726,0.095801
2,경기도 광명시,0.057613,0.264849,0.213921
3,경기도 광주시,0.063612,0.308626,0.293107
4,경기도 구리시,0.053793,0.049291,0.223461
...,...,...,...,...
180,충청북도 음성군,0.229408,0.135182,0.093839
181,충청북도 제천시,0.207172,0.238725,0.088330
182,충청북도 증평군,0.121168,0.034935,0.028592
183,충청북도 진천군,0.108735,0.133734,0.063351


In [6]:
merged_df['score'] = w1 * merged_df['소멸위험도'] + w2 * merged_df['페가지수_scaled'] + w3 * merged_df['visitors_scaled']

In [7]:
merged_df

,도시레벨,소멸위험도,페가지수_scaled,visitors_scaled,score
0,경기도 가평군,0.298097,0.275046,0.094243,2.221383
1,경기도 과천시,0.029580,0.014726,0.095801,0.096274
2,경기도 광명시,0.057613,0.264849,0.213921,0.868690
3,경기도 광주시,0.063612,0.308626,0.293107,0.950831
4,경기도 구리시,0.053793,0.049291,0.223461,0.193376
...,...,...,...,...,...
180,충청북도 음성군,0.229408,0.135182,0.093839,1.458747
181,충청북도 제천시,0.207172,0.238725,0.088330,1.663704
182,충청북도 증평군,0.121168,0.034935,0.028592,0.682053
183,충청북도 진천군,0.108735,0.133734,0.063351,0.881527


In [8]:
scaler = MinMaxScaler()
scaled_values = scaler.fit_transform(merged_df[['score']])
merged_df['final_score'] = scaled_values * 100


In [16]:
result_df = merged_df.sort_values(by='final_score', ascending=False)

result_df.columns = ['행정구역', '소멸위험', '폐가', '방문자','가중치점수','최종점수']
result_df.reset_index(drop=True, inplace=True)
result_df.to_csv('./ref_dataset/final_score.csv',index=False)
result_df


,행정구역,소멸위험,폐가,방문자,가중치점수,최종점수
0,전라남도 고흥군,0.822905,0.489156,0.031741,5.550249,100.000000
1,경상남도 합천군,0.857077,0.377696,0.027886,5.390586,97.276737
2,경상북도 의성군,0.875123,0.284812,0.033282,5.196769,93.970939
3,경상남도 남해군,0.753431,0.437554,0.033264,5.046554,91.408836
4,경상북도 청도군,0.768226,0.352002,0.035184,4.861954,88.260239
...,...,...,...,...,...,...
180,경기도 하남시,0.028447,0.022982,0.298933,-0.087752,3.836679
181,서울특별시 관악구,0.011168,0.040327,0.330703,-0.153883,2.708721
182,서울특별시 광진구,0.016400,0.032378,0.355326,-0.176191,2.328236
183,서울특별시 중구,0.046823,0.015558,0.488297,-0.207511,1.794034


In [ ]:
import plotly.graph_objects as go

def plot_result_with_dual_axis(result_df, n=10):
    # 상위 n개 행 선택 (최종점수 기준 내림차순 정렬)
    df_top = result_df.sort_values(by='최종점수', ascending=False).head(n)
    
    # x축: 행정구역
    x = df_top['행정구역']
    
    fig = go.Figure()

    # ✅ 막대그래프: 최종점수
    fig.add_trace(go.Bar(
        x=x,
        y=df_top['최종점수'],
        name='최종점수',
        yaxis='y1'
    ))

    # ✅ 보조 y축 라인그래프들
    for col in ['소멸위험', '폐가', '방문자']:
        fig.add_trace(go.Scatter(
            x=x,
            y=df_top[col],
            mode='lines+markers',
            name=col,
            yaxis='y2'
        ))

    # ✅ 레이아웃 설정
    fig.update_layout(
        title=f"상위 {n}개 지역의 최종점수 및 주요 지표 비교",
        xaxis=dict(title='행정구역'),
        yaxis=dict(title='최종점수', side='left'),
        yaxis2=dict(title='스케일링 지표 (0~1)', overlaying='y', side='right'),

    )

    fig.show()


In [21]:
plot_result_with_dual_axis(result_df,20)